In [ ]:
# notebooks/04_final_comparison.ipynb
# Sprint 4 — Comparaison finale Phase 1 : EWC, HDC, TinyOL
# Auteur : Léonard Rivals — ISAE-SUPAERO (DISC)
# Date : mai 2026
#
# Dépendances :
#   exp_001_ewc_monitoring_by_equipment  (EWC MLP — Dataset 2)
#   exp_002_hdc_monitoring_by_equipment  (HDC — Dataset 2)
#   exp_011_tinyol_monitoring_by_equipment (TinyOL FP32 — Dataset 2)
#   exp_004_tinyol_uint8                 (TinyOL UINT8 — optionnel, non disponible)

import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # backend non-interactif pour sauvegarde
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, "..")

EXP_DIR = Path("../experiments")
FIGURE_DIR = Path("../notebooks/figures/sprint4")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"EXP_DIR  : {EXP_DIR.resolve()}")
print(f"FIGURE_DIR: {FIGURE_DIR.resolve()}")

In [ ]:
# --- Cellule 1 : Chargement des résultats ---

EXPERIMENTS = {
    "EWC Online":   EXP_DIR / "exp_001_ewc_monitoring_by_equipment",
    "HDC Online":   EXP_DIR / "exp_002_hdc_monitoring_by_equipment",
    "TinyOL FP32":  EXP_DIR / "exp_011_tinyol_monitoring_by_equipment",
    "TinyOL UINT8": EXP_DIR / "exp_004_tinyol_uint8",  # non disponible → valeurs vides
}


def _load_json(path: Path) -> dict:
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}


def _get(d: dict, *keys, default="—"):
    """Navigation tolérante dans un dict imbriqué."""
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d if d is not None else default


raw: dict[str, dict] = {}
for model_name, exp_path in EXPERIMENTS.items():
    metrics = _load_json(exp_path / "results" / "metrics.json")
    memory  = _load_json(exp_path / "results" / "memory_report.json")
    raw[model_name] = {"metrics": metrics, "memory": memory, "exp_path": exp_path}
    status = "✅" if metrics else "⚠️  absent"
    print(f"  {model_name:16s} — metrics {status}")

print("\nChargement terminé.")

In [ ]:
# --- Cellule 2 : Tableau comparatif principal ---
# AA / AF / BWT / RAM / Latence / N params / Budget 256Ko
#
# Schémas JSON attendus :
#   EWC  : metrics["cl_metrics"]["ewc"][aa/af/bwt]  + memory["forward"][ram_peak_bytes/inference_latency_ms/n_params]
#   HDC  : metrics["cl_metrics"][aa/af/bwt/ram_peak_bytes/inference_latency_ms/n_params]
#   TinyOL FP32 : metrics[acc_final/avg_forgetting/backward_transfer/ram_peak_bytes/inference_latency_ms/n_params_oto+n_params_encoder]
#   TinyOL UINT8 : absent → toutes valeurs "—"

def _build_row(model_name: str) -> dict:
    m = raw[model_name]["metrics"]
    mem = raw[model_name]["memory"]

    if model_name == "EWC Online":
        aa  = _get(m, "cl_metrics", "ewc", "aa", default="—")
        af  = _get(m, "cl_metrics", "ewc", "af", default="—")
        bwt = _get(m, "cl_metrics", "ewc", "bwt", default="—")
        ram = _get(mem, "forward", "ram_peak_bytes", default="—")
        lat = _get(mem, "forward", "inference_latency_ms", default="—")
        n   = _get(mem, "forward", "n_params", default="—")

    elif model_name == "HDC Online":
        aa  = _get(m, "cl_metrics", "aa", default="—")
        af  = _get(m, "cl_metrics", "af", default="—")
        bwt = _get(m, "cl_metrics", "bwt", default="—")
        ram = _get(m, "cl_metrics", "ram_peak_bytes", default="—")
        lat = _get(m, "cl_metrics", "inference_latency_ms", default="—")
        n   = _get(m, "cl_metrics", "n_params", default="—")

    elif model_name == "TinyOL FP32":
        aa  = _get(m, "acc_final", default="—")
        af  = _get(m, "avg_forgetting", default="—")
        bwt = _get(m, "backward_transfer", default="—")
        ram = _get(m, "ram_peak_bytes", default="—")
        lat = _get(m, "inference_latency_ms", default="—")
        n_oto = _get(m, "n_params_oto", default=0)
        n_enc = _get(m, "n_params_encoder", default=0)
        n = (n_oto + n_enc) if (n_oto != "—" and n_enc != "—") else "—"

    else:  # TinyOL UINT8 — non disponible
        aa = af = bwt = ram = lat = n = "—"

    # Formatage
    def fmt_f(v, dec=4): return f"{v:.{dec}f}" if isinstance(v, float) else v
    def fmt_ram(v): return f"{v/1024:.1f} Ko" if isinstance(v, (int, float)) else v
    def fmt_lat(v): return f"{v:.3f} ms" if isinstance(v, float) else v
    budget_ok = "✅" if isinstance(ram, (int, float)) and ram < 262_144 else ("—" if ram == "—" else "❌")

    return {
        "Modèle": model_name,
        "AA": fmt_f(aa),
        "AF": fmt_f(af),
        "BWT": fmt_f(bwt),
        "RAM peak": fmt_ram(ram),
        "Latence inf.": fmt_lat(lat),
        "N params": n,
        "Budget 256Ko": budget_ok,
    }


rows = [_build_row(name) for name in EXPERIMENTS]
df = pd.DataFrame(rows).set_index("Modèle")
display(df)

In [ ]:
# --- Cellule 3 : Heatmaps des matrices d'accuracy ---

from src.evaluation.plots import plot_accuracy_matrix

TASK_NAMES = ["Pump", "Turbine", "Compressor"]

# Matrices disponibles par modèle (npy dans results/)
MATRICES = {
    "EWC Online":  raw["EWC Online"]["exp_path"] / "results" / "acc_matrix_ewc.npy",
    "HDC Online":  raw["HDC Online"]["exp_path"] / "results" / "acc_matrix_hdc.npy",
}
# TinyOL stocke la matrice dans metrics.json (liste triangulaire de longueurs variables)
tinyol_acc = _get(raw["TinyOL FP32"]["metrics"], "acc_matrix", default=None)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, mat_path) in zip(axes[:2], MATRICES.items()):
    if mat_path.exists():
        mat = np.load(mat_path)
        plot_accuracy_matrix(mat, task_names=TASK_NAMES, title=model_name, ax=ax)
    else:
        ax.set_title(f"{model_name}\n(matrice absente)")
        ax.axis("off")

# TinyOL — matrice triangulaire (liste de listes de longueurs variables)
if tinyol_acc is not None:
    T = len(TASK_NAMES)
    full = np.full((T, T), np.nan)
    rows_raw = tinyol_acc if isinstance(tinyol_acc, list) else [[tinyol_acc]]
    for i, row in enumerate(rows_raw[:T]):
        row_list = row if isinstance(row, list) else [row]
        for j, v in enumerate(row_list[:T]):
            try:
                full[i, j] = float(v)
            except (TypeError, ValueError):
                pass
    plot_accuracy_matrix(full, task_names=TASK_NAMES, title="TinyOL FP32", ax=axes[2])
else:
    axes[2].set_title("TinyOL FP32\n(matrice absente)")
    axes[2].axis("off")

plt.suptitle("Matrices d'accuracy — Phase 1 (Dataset 2, 3 tâches)", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGURE_DIR / "acc_matrices_phase1.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Sauvegardé → {FIGURE_DIR / 'acc_matrices_phase1.png'}")

In [ ]:
# --- Cellule 4 : Bar chart RAM comparatif ---

# Extraction des valeurs RAM brutes (en octets)
ram_raw = {
    "EWC Online":   _get(raw["EWC Online"]["memory"], "forward", "ram_peak_bytes"),
    "HDC Online":   _get(raw["HDC Online"]["metrics"], "cl_metrics", "ram_peak_bytes"),
    "TinyOL FP32":  _get(raw["TinyOL FP32"]["metrics"], "ram_peak_bytes"),
    "TinyOL UINT8": "—",
}

models_ram  = [k for k, v in ram_raw.items() if isinstance(v, (int, float))]
values_ko   = [ram_raw[m] / 1024 for m in models_ram]
colors      = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"][: len(models_ram)]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(models_ram, values_ko, color=colors, alpha=0.85, edgecolor="white")

ax.axhline(256, color="red",    linestyle="--", linewidth=1.5,
           label="Budget 256 Ko (NUCLEO-F439ZI)")
ax.axhline(64,  color="orange", linestyle="--", linewidth=1.5,
           label="Budget 64 Ko (STM32N6 référence)")

for bar, val in zip(bars, values_ko):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f"{val:.2f} Ko", ha="center", va="bottom", fontsize=9)

ax.set_ylabel("RAM peak inférence (Ko)")
ax.set_title("Comparaison RAM — 3 modèles CL Phase 1")
ax.legend(loc="upper right")
ax.set_ylim(0, max(values_ko) * 1.4 if values_ko else 30)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "ram_comparison.png", dpi=150)
plt.show()
print(f"Sauvegardé → {FIGURE_DIR / 'ram_comparison.png'}")

In [ ]:
# --- Cellule 5 : Synthèse Triple Gap ---

# Valeurs mesurées
ewc_ram_ko  = ram_raw["EWC Online"] / 1024  if isinstance(ram_raw["EWC Online"], (int,float)) else None
hdc_ram_ko  = ram_raw["HDC Online"] / 1024  if isinstance(ram_raw["HDC Online"], (int,float)) else None
tol_ram_ko  = ram_raw["TinyOL FP32"] / 1024 if isinstance(ram_raw["TinyOL FP32"], (int,float)) else None

gap2_vals = " · ".join([
    f"EWC {ewc_ram_ko:.1f} Ko" if ewc_ram_ko else "EWC —",
    f"HDC {hdc_ram_ko:.1f} Ko" if hdc_ram_ko else "HDC —",
    f"TinyOL {tol_ram_ko:.1f} Ko" if tol_ram_ko else "TinyOL —",
])

gap_summary = pd.DataFrame({
    "Gap": [
        "Gap 1 — Données industrielles réelles",
        "Gap 2 — RAM < 100 Ko mesurée (NUCLEO-F439ZI 256Ko)",
        "Gap 3 — Quantif. INT8 en CL",
    ],
    "Statut": [
        "⚠️  Partiellement (Dataset 2 — Monitoring équipements industriels)",
        f"✅  {gap2_vals}",
        "🔄  En cours — ewc_mlp_int8.onnx + buffer UINT8 TinyOL (exp_004)",
    ],
    "Exp. clé": [
        "exp_001, exp_002, exp_011",
        "exp_001 (<1% budget 256Ko), exp_002, exp_011",
        "experiments/onnx_sprint4/ewc_mlp_int8.onnx",
    ],
})

display(gap_summary)

## Conclusion Phase 1

### Résultats clés

| Modèle | Points forts | Points faibles |
|--------|-------------|----------------|
| **M2 EWC Online** | Meilleure précision (AA≈0.98), RAM inférence minimale (1.1 Ko), oubli quasi-nul (AF≈0.001) | RAM update plus élevée (27 Ko) due à la backprop |
| **M3 HDC Online** | AF=0 par construction (mémoire additive), simple à porter sur MCU | RAM prototype (14 Ko), AA plus faible (≈0.87) |
| **M1 TinyOL FP32** | RAM update OtO très faible (6.2 Ko backbone gelé), AF≈0.008 | AA modéré (≈0.91) ; backbone 25→8 surdimensionné pour 4 features |

### Recommandation Phase 2

**Priorité 1 — Porter M2 (EWC MLP)** : meilleur compromis précision/RAM. L'export ONNX est validé (`experiments/onnx_sprint4/ewc_mlp.onnx`, max|Δ|=1.2e-7). Portage C sur NUCLEO-F439ZI : 705 params × 4 B = 2.8 Ko en Flash.

**Priorité 2 — Porter M3 (HDC)** : AF=0 garanti, 14 Ko SRAM, portage direct via POPCOUNT. Valeur différenciante pour le Gap 2 (aucun gradient requis).

**Priorité 3 — M1 TinyOL** : backbone gelé + OtO 40 octets. Utile si données temporelles (Dataset 1 — Pump).

### Prochaines étapes (Sprint 10+)
- Portage C sur NUCLEO-F439ZI (Cortex-M4, 256 Ko SRAM)
- Mesures DWT latence réelle (objectif : < 100 ms par inférence+update)
- Validation MCU sur données streaming Dataset 2

---
**TODO(arnaud)**: Inclure tableau comparaison Dataset 1 vs Dataset 2 dans ce notebook ou réserver pour le manuscrit ?  
**TODO(fred)**: Quels critères Edge Spectrum prioritise pour le choix du modèle à déployer (précision vs RAM vs latence) ?